In [0]:
# Import the required Delta Lake and PySpark components

from delta.tables import DeltaTable

from pyspark.sql.functions import (
    col,
    current_timestamp,
    lit
)

In [0]:
# Receive job parameters

dbutils.widgets.text(
    "batch_id",
    "2009-12",
    "Batch ID"
)

dbutils.widgets.text(
    "run_id",
    "manual-run-001",
    "Run ID"
)


# Receive upstream task result states and error codes

dbutils.widgets.text(
    "bronze_status",
    "",
    "Bronze Status"
)

dbutils.widgets.text(
    "bronze_error",
    "",
    "Bronze Error"
)

dbutils.widgets.text(
    "silver_status",
    "",
    "Silver Status"
)

dbutils.widgets.text(
    "silver_error",
    "",
    "Silver Error"
)

dbutils.widgets.text(
    "gold_status",
    "",
    "Gold Status"
)

dbutils.widgets.text(
    "gold_error",
    "",
    "Gold Error"
)

dbutils.widgets.text(
    "validation_status",
    "",
    "Validation Status"
)

dbutils.widgets.text(
    "validation_error",
    "",
    "Validation Error"
)


# Read the supplied parameter values

batch_id = dbutils.widgets.get("batch_id")
run_id = dbutils.widgets.get("run_id")

bronze_status = dbutils.widgets.get("bronze_status")
bronze_error = dbutils.widgets.get("bronze_error")

silver_status = dbutils.widgets.get("silver_status")
silver_error = dbutils.widgets.get("silver_error")

gold_status = dbutils.widgets.get("gold_status")
gold_error = dbutils.widgets.get("gold_error")

validation_status = dbutils.widgets.get(
    "validation_status"
)

validation_error = dbutils.widgets.get(
    "validation_error"
)

control_table = "online_retail.control.pipeline_runs"


print(f"Run ID: {run_id}")
print(f"Batch ID: {batch_id}")

print(
    f"Bronze: {bronze_status} | "
    f"Error: {bronze_error}"
)

print(
    f"Silver: {silver_status} | "
    f"Error: {silver_error}"
)

print(
    f"Gold: {gold_status} | "
    f"Error: {gold_error}"
)

print(
    f"Validation: {validation_status} | "
    f"Error: {validation_error}"
)

In [0]:
# Normalize Lakeflow task states for the control table

def normalize_status(task_status):
    return (
        task_status
        .strip()
        .upper()
        .replace(" ", "_")
        .replace("-", "_")
    )


task_results = [
    ("bronze", bronze_status, bronze_error),
    ("silver", silver_status, silver_error),
    ("gold", gold_status, gold_error),
    (
        "validation",
        validation_status,
        validation_error
    )
]


# Retain only tasks that did not complete successfully

failed_task_results = []

for layer_name, task_status, error_code in task_results:
    normalized_status = normalize_status(task_status)

    if normalized_status and normalized_status != "SUCCESS":
        error_message = (
            error_code.strip()
            if error_code.strip()
            else (
                f"Task ended with status "
                f"{normalized_status}"
            )
        )

        failed_task_results.append(
            (
                layer_name,
                normalized_status,
                error_message
            )
        )


if not failed_task_results:
    raise ValueError(
        "The failure handler ran without receiving any "
        "unsuccessful task results."
    )


print(
    f"Unsuccessful task records received: "
    f"{len(failed_task_results)}"
)

for task_result in failed_task_results:
    print(task_result)

In [0]:
# Write failed and skipped task results to the control table

if not spark.catalog.tableExists(control_table):
    raise ValueError(
        f"Control table does not exist: {control_table}"
    )


failure_updates_df = (
    spark.createDataFrame(
        failed_task_results,
        """
        layer_name STRING,
        status STRING,
        error_message STRING
        """
    )
    .withColumn(
        "run_id",
        lit(run_id)
    )
    .withColumn(
        "batch_id",
        lit(batch_id)
    )
    .withColumn(
        "start_timestamp",
        lit(None).cast("timestamp")
    )
    .withColumn(
        "end_timestamp",
        current_timestamp()
    )
    .withColumn(
        "input_row_count",
        lit(None).cast("long")
    )
    .withColumn(
        "output_row_count",
        lit(None).cast("long")
    )
    .select(
        "run_id",
        "batch_id",
        "layer_name",
        "status",
        "start_timestamp",
        "end_timestamp",
        "input_row_count",
        "output_row_count",
        "error_message"
    )
)


control_delta_table = DeltaTable.forName(
    spark,
    control_table
)


(
    control_delta_table.alias("target")
    .merge(
        failure_updates_df.alias("source"),
        """
        target.run_id = source.run_id
        AND target.batch_id = source.batch_id
        AND target.layer_name = source.layer_name
        """
    )
    .whenMatchedUpdate(
        set={
            "status": "source.status",
            "end_timestamp": "source.end_timestamp",
            "error_message": "source.error_message"
        }
    )
    .whenNotMatchedInsertAll()
    .execute()
)


print(
    f"Recorded {len(failed_task_results)} unsuccessful "
    f"task results for run {run_id}."
)

In [0]:
# Display the control records for the failed pipeline run

current_run_audit_df = (
    spark.table(control_table)
    .filter(
        (col("run_id") == run_id)
        & (col("batch_id") == batch_id)
    )
    .orderBy(
        "start_timestamp",
        "layer_name"
    )
)

current_run_audit_df.show(
    truncate=False
)

In [0]:
# Raise an exception after logging so the overall job remains failed

failure_summary = ", ".join(
    f"{layer_name}={status}"
    for layer_name, status, error_message
    in failed_task_results
)


raise RuntimeError(
    f"Pipeline run {run_id} failed for batch {batch_id}. "
    f"Unsuccessful layers: {failure_summary}."
)